# IterationVIT - Tea Pest Classification
This notebook implements the IterationVIT model for tea pest classification, based on the paper:
**"Study on the Tea Pest Classification Model Using a Convolutional and Embedded Iterative Region of Interest Encoding Transformer"** (Zhan et al., 2023)

In [1]:
# Install dependencies (uncomment if needed)
# !pip install einops
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [2]:
# Bottleneck Convolution Module
class BottleneckConv(nn.Module):
    def __init__(self, in_channels, bottleneck_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, bottleneck_channels, kernel_size=1),
            nn.BatchNorm2d(bottleneck_channels),
            nn.ReLU(),
            nn.Conv2d(bottleneck_channels, bottleneck_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(bottleneck_channels),
            nn.ReLU(),
            nn.Conv2d(bottleneck_channels, in_channels, kernel_size=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU()
        )

    def forward(self, x):
        return self.block(x)

In [3]:
# Iterative ROI Encoding Module
class IterativeROI(nn.Module):
    def __init__(self, num_iters, embed_dim, num_patches):
        super().__init__()
        self.num_iters = num_iters
        self.linear = nn.Linear(2, embed_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(embed_dim, nhead=8, dim_feedforward=512),
            num_layers=4
        )
        self.num_patches = num_patches

    def forward(self, x):
        B, C, H, W = x.shape
        out = rearrange(x, 'b c h w -> b (h w) c')
        pos = torch.rand(B, self.num_patches**2, 2).to(x.device)
        pos_embed = self.linear(pos)
        tokens = out[:, :self.num_patches**2, :] + pos_embed
        tokens = self.transformer(tokens)
        return tokens

In [4]:
# IterationVIT Full Model
class IterationVIT(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.conv = BottleneckConv(3, 64)
        self.roi_encoder = IterativeROI(num_iters=3, embed_dim=64, num_patches=16)
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.conv(x)
        x = self.roi_encoder(x)
        x = x.mean(dim=1)
        return self.classifier(x)